# Classical Router Training (CE + KL)

This notebook trains a VLM router using **classical multi-class classification** approach.

## Approach

Standard classification with both hard and soft labels:
- **Hard labels**: Best model per (sample, mode) - used for Cross-Entropy (CE) loss
- **Soft labels**: Probability distribution over models from teacher - used for KL divergence
- **Combined loss**: L = α * CE + (1-α) * KL

## Benefits

- **Simple & interpretable**: Standard classification framework
- **Calibrated probabilities**: Softmax outputs give model selection confidence
- **Knowledge distillation**: Soft labels capture relative model performance
- **Well-studied**: Extensive literature on classification + distillation

## Sections

1. Setup & Configuration
2. Load Data from SQL
3. Compute Rewards & Create Labels
4. Build Classification Dataset
5. Train Classical Router
6. Evaluate & Compare to Oracle
7. Visualizations

## 1. Setup & Configuration

In [ ]:
# Add parent directory to path
import sys
import os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import logging
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# Local imports
from config import DBConfig, RewardWeights
from db_utils import load_profiles_real_schema, test_connection
from reward_definitions import compute_rewards_real_schema
from models.classical_router import ClassicalRouterModel, create_soft_labels_from_metrics

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Configuration
LIMIT = 1000  # Set to None for full dataset, or use small number for testing
DATA_SPLIT = "train"  # Load only training data

# Loss settings
TEMPERATURE = 2.0  # Temperature for soft labels (higher = softer)
ALPHA = 0.5  # Weight for CE loss (1-alpha for KL loss)

# Training settings
BATCH_SIZE = 16
NUM_EPOCHS = 5
LEARNING_RATE = 2e-5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Paths
DATA_DIR = Path("../data")
MODELS_DIR = Path("../models/checkpoints")
DATA_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True, parents=True)

print("Configuration:")
print(f"  LIMIT: {LIMIT}")
print(f"  DATA_SPLIT: {DATA_SPLIT}")
print(f"  TEMPERATURE: {TEMPERATURE}")
print(f"  ALPHA (CE weight): {ALPHA}")
print(f"  BATCH_SIZE: {BATCH_SIZE}")
print(f"  NUM_EPOCHS: {NUM_EPOCHS}")
print(f"  LEARNING_RATE: {LEARNING_RATE}")
print(f"  DEVICE: {DEVICE}")

## 2. Load Data from SQL

In [ ]:
# Load database configuration
db_config = DBConfig.from_env()

# Test connection
print("Testing database connection...")
if test_connection(db_config):
    print("✓ Database connection successful")
else:
    raise RuntimeError("Database connection failed. Check credentials.")

In [ ]:
# Load profiling data from SQL
print(f"\nLoading profiling data (data_split={DATA_SPLIT}, limit={LIMIT})...")

df_profiles = load_profiles_real_schema(
    db_config=db_config,
    limit=LIMIT,
    data_split=DATA_SPLIT,
)

print(f"\n✓ Loaded {len(df_profiles)} profile records")
print(f"  Unique samples: {df_profiles['sample_id'].nunique()}")
print(f"  Unique models: {df_profiles['model_name'].nunique()}")
print(f"  Models: {sorted(df_profiles['model_name'].unique())}")

# Show sample
df_profiles.head()

## 3. Compute Rewards & Create Labels

In [ ]:
# Compute multi-objective rewards
print("\nComputing rewards...")

reward_weights = RewardWeights()  # Use defaults
df_with_rewards = compute_rewards_real_schema(df_profiles, reward_weights)

print(f"✓ Computed rewards for {len(df_with_rewards)} records")
print("\nReward columns:")
reward_cols = [c for c in df_with_rewards.columns if c.startswith('reward_')]
print(reward_cols)

# Show reward statistics
print("\nReward statistics:")
print(df_with_rewards[reward_cols].describe())

In [ ]:
# Expand to include all 4 modes
print("\nExpanding dataset to include all 4 routing modes...")

modes = ['accuracy', 'cheap', 'fast', 'balanced']
expanded_rows = []

for _, row in df_with_rewards.iterrows():
    for mode in modes:
        row_dict = row.to_dict()
        row_dict['mode_id'] = mode
        row_dict['reward'] = row[f'reward_{mode}']
        expanded_rows.append(row_dict)

df_expanded = pd.DataFrame(expanded_rows)

print(f"✓ Expanded to {len(df_expanded)} rows ({len(df_with_rewards)} → {len(df_expanded)})")
print(f"  Rows per mode: {len(df_expanded) // 4}")

df_expanded.head()

In [ ]:
# Create model and mode mappings
unique_models = sorted(df_expanded['model_name'].unique())
unique_modes = sorted(df_expanded['mode_id'].unique())

model_to_id = {model: i for i, model in enumerate(unique_models)}
id_to_model = {i: model for model, i in model_to_id.items()}

mode_to_id = {mode: i for i, mode in enumerate(unique_modes)}
id_to_mode = {i: mode for mode, i in mode_to_id.items()}

print(f"Models ({len(model_to_id)}): {model_to_id}")
print(f"Modes ({len(mode_to_id)}): {mode_to_id}")

# Save mappings
with open(DATA_DIR / "model_index_classical.json", "w") as f:
    json.dump(model_to_id, f, indent=2)
with open(DATA_DIR / "mode_index_classical.json", "w") as f:
    json.dump(mode_to_id, f, indent=2)

print("\n✓ Saved model and mode index mappings")

In [ ]:
# Create per-(sample, mode) dataset with hard labels
print("\nCreating per-sample dataset with hard labels...")

# Group by (sample_id, mode_id) and find best model
per_sample_data = []

grouped = df_expanded.groupby(['sample_id', 'mode_id'])

for (sample_id, mode_id), group in grouped:
    # Best model (hard label)
    best_idx = group['reward'].idxmax()
    best_model = group.loc[best_idx, 'model_name']
    best_reward = group.loc[best_idx, 'reward']
    
    # Sample metadata (same for all models)
    sample_row = group.iloc[0]
    
    per_sample_data.append({
        'sample_id': sample_id,
        'mode_id': mode_id,
        'best_model': best_model,
        'best_reward': best_reward,
        'prompt_raw': sample_row['prompt_raw'],
        'txt_prompt_length_chars': sample_row.get('txt_prompt_length_chars'),
        'txt_prompt_length_words': sample_row.get('txt_prompt_length_words'),
        'img_width': sample_row.get('img_width'),
        'img_height': sample_row.get('img_height'),
        'img_aspect_ratio': sample_row.get('img_aspect_ratio'),
        'source_dataset': sample_row.get('source_dataset'),
        'router_task': sample_row.get('router_task'),
        'data_split': sample_row.get('data_split'),
    })

df_per_sample = pd.DataFrame(per_sample_data)

print(f"✓ Created per-sample dataset: {len(df_per_sample)} examples")
print(f"  Unique samples: {df_per_sample['sample_id'].nunique()}")
print(f"  Examples per mode: {len(df_per_sample) // len(modes)}")

df_per_sample.head()

In [ ]:
# Create soft labels from reward distributions
print(f"\nCreating soft labels (temperature={TEMPERATURE})...")

soft_labels = create_soft_labels_from_metrics(
    df_per_sample=df_expanded,  # Need all (sample, mode, model) rows
    metric_column='reward',
    temperature=TEMPERATURE,
    model_to_id=model_to_id,
)

print(f"✓ Created soft labels: shape {soft_labels.shape}")
print(f"  Expected shape: ({len(df_per_sample)}, {len(model_to_id)})")

# Verify soft labels are valid probability distributions
print(f"\nSoft label statistics:")
print(f"  Row sums (should be ~1.0): {soft_labels.sum(dim=1).mean():.6f} ± {soft_labels.sum(dim=1).std():.6f}")
print(f"  Min value: {soft_labels.min():.6f}")
print(f"  Max value: {soft_labels.max():.6f}")

# Show example soft label distribution
print(f"\nExample soft label distribution:")
print(soft_labels[0])

## 4. Build Classification Dataset

In [ ]:
# Create PyTorch dataset
class ClassificationDataset(Dataset):
    def __init__(self, df_per_sample, soft_labels, model_to_id, mode_to_id):
        self.df = df_per_sample.reset_index(drop=True)
        self.soft_labels = soft_labels
        self.model_to_id = model_to_id
        self.mode_to_id = mode_to_id
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Build input text
        parts = [f"Prompt: {row['prompt_raw']}"]
        if pd.notna(row.get('txt_prompt_length_chars')):
            parts.append(f"PromptLen: {int(row['txt_prompt_length_chars'])} chars")
        if pd.notna(row.get('img_width')) and pd.notna(row.get('img_height')):
            parts.append(f"Image: {int(row['img_width'])}x{int(row['img_height'])}")
        if pd.notna(row.get('source_dataset')):
            parts.append(f"Dataset: {row['source_dataset']}")
        if pd.notna(row.get('router_task')):
            parts.append(f"Task: {row['router_task']}")
        
        sample_text = " | ".join(parts)
        
        # Hard label
        hard_label = self.model_to_id[row['best_model']]
        
        # Soft label
        soft_label = self.soft_labels[idx]
        
        # Mode ID
        mode_id = self.mode_to_id[row['mode_id']]
        
        return sample_text, mode_id, hard_label, soft_label

# Create dataset
train_dataset = ClassificationDataset(
    df_per_sample=df_per_sample,
    soft_labels=soft_labels,
    model_to_id=model_to_id,
    mode_to_id=mode_to_id,
)

print(f"✓ Created training dataset: {len(train_dataset)} examples")

In [ ]:
# Create collate function
def collate_classification_batch(batch):
    sample_texts, mode_ids, hard_labels, soft_labels = zip(*batch)
    
    return {
        'sample_texts': list(sample_texts),
        'mode_ids': torch.tensor(mode_ids, dtype=torch.long),
        'hard_labels': torch.tensor(hard_labels, dtype=torch.long),
        'soft_labels': torch.stack(soft_labels),
    }

# Create dataloader
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_classification_batch,
)

print(f"✓ Created training dataloader: {len(train_loader)} batches")

# Test batch
sample_batch = next(iter(train_loader))
print("\nSample batch keys:", sample_batch.keys())
print("  sample_texts:", len(sample_batch['sample_texts']))
print("  mode_ids:", sample_batch['mode_ids'].shape)
print("  hard_labels:", sample_batch['hard_labels'].shape)
print("  soft_labels:", sample_batch['soft_labels'].shape)

## 5. Train Classical Router

In [ ]:
# Initialize model
model = ClassicalRouterModel(
    num_models=len(model_to_id),
    num_modes=len(mode_to_id),
    text_encoder_name="distilbert-base-uncased",
    mode_embed_dim=16,
    hidden_dim=256,
    dropout=0.1,
)

model = model.to(DEVICE)

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

print(f"\n✓ Initialized model on {DEVICE}")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Training loop
print(f"\nTraining for {NUM_EPOCHS} epochs...\n")

history = {
    'train_loss': [],
    'ce_loss': [],
    'kl_loss': [],
    'accuracy': [],
}

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    epoch_ce = 0.0
    epoch_kl = 0.0
    epoch_correct = 0
    epoch_total = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    
    for batch in pbar:
        optimizer.zero_grad()
        
        # Move to device
        mode_ids = batch['mode_ids'].to(DEVICE)
        hard_labels = batch['hard_labels'].to(DEVICE)
        soft_labels = batch['soft_labels'].to(DEVICE)
        
        # Forward pass
        loss, loss_dict = model.compute_loss(
            sample_texts=batch['sample_texts'],
            mode_ids=mode_ids,
            hard_labels=hard_labels,
            soft_labels=soft_labels,
            temperature=TEMPERATURE,
            alpha=ALPHA,
        )
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Track metrics
        epoch_loss += loss_dict['total_loss']
        epoch_ce += loss_dict['ce_loss']
        epoch_kl += loss_dict.get('kl_loss', 0.0)
        
        # Compute accuracy
        with torch.no_grad():
            predictions, _ = model.predict(batch['sample_texts'], mode_ids)
            correct = (predictions == hard_labels).sum().item()
            epoch_correct += correct
            epoch_total += len(hard_labels)
        
        pbar.set_postfix({
            'loss': f"{loss_dict['total_loss']:.4f}",
            'ce': f"{loss_dict['ce_loss']:.4f}",
            'kl': f"{loss_dict.get('kl_loss', 0.0):.4f}",
            'acc': f"{100 * correct / len(hard_labels):.1f}%",
        })
    
    # Epoch statistics
    avg_loss = epoch_loss / len(train_loader)
    avg_ce = epoch_ce / len(train_loader)
    avg_kl = epoch_kl / len(train_loader)
    accuracy = 100 * epoch_correct / epoch_total
    
    history['train_loss'].append(avg_loss)
    history['ce_loss'].append(avg_ce)
    history['kl_loss'].append(avg_kl)
    history['accuracy'].append(accuracy)
    
    print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}, CE = {avg_ce:.4f}, KL = {avg_kl:.4f}, Acc = {accuracy:.2f}%")

print("\n✓ Training complete")

In [ ]:
# Save trained model
model_path = MODELS_DIR / "best_classical_router.pt"
torch.save({
    'model_state_dict': model.state_dict(),
    'model_to_id': model_to_id,
    'mode_to_id': mode_to_id,
    'config': {
        'num_models': len(model_to_id),
        'num_modes': len(mode_to_id),
        'text_encoder_name': 'distilbert-base-uncased',
        'mode_embed_dim': 16,
        'hidden_dim': 256,
        'dropout': 0.1,
    },
}, model_path)

print(f"✓ Saved model to {model_path}")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Total loss
axes[0].plot(history['train_loss'], marker='o', label='Total Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Total Training Loss')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# CE vs KL
axes[1].plot(history['ce_loss'], marker='o', label='CE Loss', color='blue')
axes[1].plot(history['kl_loss'], marker='s', label='KL Loss', color='orange')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('CE vs KL Loss')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# Accuracy
axes[2].plot(history['accuracy'], marker='o', color='green', label='Accuracy')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Accuracy (%)')
axes[2].set_title('Training Accuracy')
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

## 6. Evaluate & Compare to Oracle

In [ ]:
# Load validation data
print("Loading validation data...")

df_val_profiles = load_profiles_real_schema(
    db_config=db_config,
    limit=500,  # Smaller validation set
    data_split="val",
)

# Compute rewards
df_val_with_rewards = compute_rewards_real_schema(df_val_profiles, reward_weights)

# Expand modes
val_expanded_rows = []
for _, row in df_val_with_rewards.iterrows():
    for mode in modes:
        row_dict = row.to_dict()
        row_dict['mode_id'] = mode
        row_dict['reward'] = row[f'reward_{mode}']
        val_expanded_rows.append(row_dict)

df_val_expanded = pd.DataFrame(val_expanded_rows)

print(f"✓ Loaded {len(df_val_expanded)} validation rows")
print(f"  Unique samples: {df_val_expanded['sample_id'].nunique()}")

In [ ]:
# Evaluate routing accuracy vs oracle
print("\nEvaluating routing accuracy...\n")

model.eval()
results = []

# Group by (sample_id, mode_id)
grouped = df_val_expanded.groupby(['sample_id', 'mode_id'])

for (sample_id, mode_id), group in tqdm(grouped, desc="Evaluating"):
    # Oracle choice (best reward)
    oracle_choice = group.loc[group['reward'].idxmax(), 'model_name']
    oracle_reward = group['reward'].max()
    
    # Router prediction
    sample_text = group.iloc[0]['prompt_raw']  # Same for all models
    
    # Build input text
    row = group.iloc[0]
    parts = [f"Prompt: {row['prompt_raw']}"]
    if pd.notna(row.get('txt_prompt_length_chars')):
        parts.append(f"PromptLen: {int(row['txt_prompt_length_chars'])} chars")
    if pd.notna(row.get('img_width')) and pd.notna(row.get('img_height')):
        parts.append(f"Image: {int(row['img_width'])}x{int(row['img_height'])}")
    if pd.notna(row.get('source_dataset')):
        parts.append(f"Dataset: {row['source_dataset']}")
    if pd.notna(row.get('router_task')):
        parts.append(f"Task: {row['router_task']}")
    sample_text = " | ".join(parts)
    
    with torch.no_grad():
        predictions, probs = model.predict(
            sample_texts=[sample_text],
            mode_ids=torch.tensor([mode_to_id[mode_id]], dtype=torch.long),
        )
        predicted_model_id = predictions[0].item()
        predicted_model = id_to_model[predicted_model_id]
        prediction_confidence = probs[0, predicted_model_id].item()
    
    # Get router's reward
    router_reward = group.loc[group['model_name'] == predicted_model, 'reward'].values[0]
    
    results.append({
        'sample_id': sample_id,
        'mode_id': mode_id,
        'oracle_choice': oracle_choice,
        'oracle_reward': oracle_reward,
        'router_choice': predicted_model,
        'router_reward': router_reward,
        'confidence': prediction_confidence,
        'correct': oracle_choice == predicted_model,
        'reward_gap': oracle_reward - router_reward,
    })

df_results = pd.DataFrame(results)

# Print results
overall_acc = 100 * df_results['correct'].mean()
avg_gap = df_results['reward_gap'].mean()
median_gap = df_results['reward_gap'].median()
avg_confidence = df_results['confidence'].mean()

print("\n" + "="*60)
print("EVALUATION RESULTS")
print("="*60)
print(f"Routing Accuracy: {overall_acc:.2f}%")
print(f"Average Confidence: {avg_confidence:.4f}")
print(f"Average Reward Gap: {avg_gap:.4f}")
print(f"Median Reward Gap: {median_gap:.4f}")
print(f"Oracle Reward (avg): {df_results['oracle_reward'].mean():.4f}")
print(f"Router Reward (avg): {df_results['router_reward'].mean():.4f}")
print("="*60)

# Per-mode breakdown
print("\nPer-mode breakdown:")
for mode in modes:
    mode_results = df_results[df_results['mode_id'] == mode]
    mode_acc = 100 * mode_results['correct'].mean()
    mode_gap = mode_results['reward_gap'].mean()
    mode_conf = mode_results['confidence'].mean()
    print(f"  {mode:12s}: Acc = {mode_acc:5.2f}%, Gap = {mode_gap:.4f}, Conf = {mode_conf:.4f}")

## 7. Visualizations

In [ ]:
# Routing accuracy by mode
mode_acc = df_results.groupby('mode_id')['correct'].mean() * 100

plt.figure(figsize=(10, 5))
plt.bar(mode_acc.index, mode_acc.values, color='steelblue', edgecolor='black', alpha=0.7)
plt.xlabel('Mode')
plt.ylabel('Routing Accuracy (%)')
plt.title('Routing Accuracy by Mode')
plt.axhline(overall_acc, color='red', linestyle='--', label=f'Overall = {overall_acc:.2f}%')
plt.ylim([0, 100])
plt.grid(True, alpha=0.3, axis='y')
plt.legend()
plt.show()

In [ ]:
# Confidence vs correctness
correct_conf = df_results[df_results['correct'] == True]['confidence']
incorrect_conf = df_results[df_results['correct'] == False]['confidence']

plt.figure(figsize=(10, 5))
plt.hist(correct_conf, bins=30, alpha=0.7, label='Correct', color='green', edgecolor='black')
plt.hist(incorrect_conf, bins=30, alpha=0.7, label='Incorrect', color='red', edgecolor='black')
plt.xlabel('Prediction Confidence')
plt.ylabel('Count')
plt.title('Prediction Confidence Distribution')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Average confidence (correct): {correct_conf.mean():.4f}")
print(f"Average confidence (incorrect): {incorrect_conf.mean():.4f}")

In [ ]:
# Reward gap distribution
plt.figure(figsize=(10, 5))
plt.hist(df_results['reward_gap'], bins=50, edgecolor='black', alpha=0.7, color='coral')
plt.xlabel('Reward Gap (Oracle - Router)')
plt.ylabel('Count')
plt.title('Distribution of Reward Gap')
plt.axvline(0, color='green', linestyle='--', linewidth=2, label='Perfect routing (gap=0)')
plt.axvline(avg_gap, color='red', linestyle='--', label=f'Mean gap = {avg_gap:.4f}')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

perfect_routing = (df_results['reward_gap'] == 0).mean() * 100
print(f"\nPerfect routing rate: {perfect_routing:.2f}%")

In [ ]:
# Oracle vs Router rewards scatter
plt.figure(figsize=(8, 8))
plt.scatter(
    df_results['oracle_reward'],
    df_results['router_reward'],
    alpha=0.5,
    s=30,
    c=df_results['correct'].map({True: 'green', False: 'red'}),
)
plt.plot([0, 1], [0, 1], 'k--', label='Perfect routing')
plt.xlabel('Oracle Reward')
plt.ylabel('Router Reward')
plt.title('Oracle vs Router Rewards')
plt.grid(True, alpha=0.3)
plt.legend(['Perfect routing', 'Correct choice', 'Incorrect choice'])
plt.axis('equal')
plt.show()

In [ ]:
# Model selection distribution
oracle_counts = df_results['oracle_choice'].value_counts()
router_counts = df_results['router_choice'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Oracle
axes[0].bar(range(len(oracle_counts)), oracle_counts.values, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xticks(range(len(oracle_counts)))
axes[0].set_xticklabels(oracle_counts.index, rotation=45, ha='right')
axes[0].set_ylabel('Count')
axes[0].set_title('Oracle Model Selection')
axes[0].grid(True, alpha=0.3, axis='y')

# Router
axes[1].bar(range(len(router_counts)), router_counts.values, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_xticks(range(len(router_counts)))
axes[1].set_xticklabels(router_counts.index, rotation=45, ha='right')
axes[1].set_ylabel('Count')
axes[1].set_title('Router Model Selection')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated training a **classical classification router** with CE + KL loss:

### Key Results
- **Training Accuracy**: How well the model predicts hard labels during training
- **Routing Accuracy**: How often the router picks the same model as the oracle
- **Prediction Confidence**: Softmax probability of the predicted model
- **Reward Gap**: Difference between oracle and router rewards

### Comparison to Other Approaches
- **Reward-based (Notebook 02)**: Predicts scalar rewards, then picks highest
- **Pairwise ranking (Notebook 03)**: Learns relative preferences between models
- **Classical (this notebook)**: Direct classification with soft label distillation

### Next Steps
1. **Compare all 3 approaches**: Evaluate on same test set
2. **Tune hyperparameters**: Adjust temperature, alpha, learning rate
3. **Ensemble methods**: Combine multiple router approaches
4. **Production deployment**: Use trained model for real-time routing

### Files Generated
- `../models/checkpoints/best_classical_router.pt` - Trained model
- `../data/model_index_classical.json` - Model ID mappings
- `../data/mode_index_classical.json` - Mode ID mappings